In [ ]:
chembl_full_path = "data/chembl_35_cleaned.csv"
chembl_filtered_path = "data/chembl_35_cleaned_PAINS_covalent.csv"
coconut_path = "data/coconut.smi"

# coconut.smi
script to convert coconut sdf to RDKit parsable smiles

In [ ]:
from rdkit import Chem
from rdkit import RDLogger
import csv
RDLogger.DisableLog('rdApp.*')
from tqdm import tqdm

COCONUT_SDF_PATH = "/path/to/coconut_sdf_2d_lite-03-2026.sdf"
suppl = Chem.SDMolSupplier(COCONUT_SDF_PATH)
coco_smis = []
for m in tqdm(suppl):
    if m:
        try:
            smi = Chem.MolToSmiles(m)
            coco_smis.append(smi)
        except:
            print("conversion failure")

with open("coconut.smi", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    for smi in coco_smis:
        writer.writerow([smi])

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from rdkit import Chem
from lacan import lacan


def init_worker():
    global profile_full, profile_filtered, profile_coconut
    profile_full = lacan.load_profile("chembl_full")
    profile_filtered = lacan.load_profile("chembl")
    profile_coconut = lacan.load_profile("coconut")


def get_scores_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [0,0,0]

    try:
        return [
            lacan.score_mol(mol, profile_full)[0],
            lacan.score_mol(mol, profile_filtered)[0],
            lacan.score_mol(mol, profile_coconut)[0],
        ]
    except Exception as e:
        print("FAILURE", e, smiles)
        return [0,0,0]


def score_file(path, n_workers=20):

    with open(path) as f:
        smiles_iter = (line.strip() for line in f if line.strip())

        with ProcessPoolExecutor(
            max_workers=n_workers,
            initializer=init_worker
        ) as executor:

            results = list(
                tqdm(
                    executor.map(get_scores_from_smiles, smiles_iter, chunksize=256),
                )
            )

    return results


chembl_full_scores = score_file(chembl_full_path)
chembl_filtered_scores = score_file(chembl_filtered_path)
coconut_scores = score_file(coconut_path)

In [ ]:
# --- Load and score ChEMBL36 new sets ---
import pandas as pd
chembl36_full_smiles     = pd.read_csv("data/chembl36_new_full.csv", header=None)[0].dropna().tolist()
chembl36_filtered_smiles = pd.read_csv("data/chembl36_new_filtered.csv", header=None)[0].dropna().tolist()

chembl36_full_scores     = score_file("data/chembl36_new_full.csv")
chembl36_filtered_scores = score_file("data/chembl36_new_filtered.csv")

chembl_full_smiles     = pd.read_csv("data/chembl_35_cleaned.csv", header=None)[0].dropna().tolist()
chembl_filtered_smiles = pd.read_csv("data/chembl_35_cleaned_PAINS_covalent.csv", header=None)[0].dropna().tolist()
coconut_smiles     = pd.read_csv("data/coconut.smi", header=None)[0].dropna().tolist()


In [ ]:
!pip install seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw
from io import BytesIO
from PIL import Image

sns.set_style("white")
sns.set_context("paper", font_scale=1.1)



# --- Dataset registry ---
datasets = {
    "ChEMBL Full":        (np.array(chembl_full_scores),     None),
    "ChEMBL Filtered":    (np.array(chembl_filtered_scores), None),
    "COCONUT":            (np.array(coconut_scores),         None),
    "ChEMBL36 Full":      (np.array(chembl36_full_scores),   None),
    "ChEMBL36 Filtered":  (np.array(chembl36_filtered_scores), None),
}

# Store SMILES alongside scores for rejection visualization (diagonal only)
diag_smiles = {
    "ChEMBL Full":     chembl_full_smiles,        # you need to have these as lists
    "ChEMBL Filtered": chembl_filtered_smiles,
    "COCONUT":         coconut_smiles,
}

profile_names = ["ChEMBL Full", "ChEMBL Filtered", "COCONUT"]
profile_col   = [0, 1, 2]

COLORS = {
    "ChEMBL Full":       "#4878CF",
    "ChEMBL Filtered":   "#6ACC65",
    "COCONUT":           "#D65F5F",
    "ChEMBL36 Full":     "#4878CF",   # same color family as ChEMBL Full
    "ChEMBL36 Filtered": "#6ACC65",   # same color family as ChEMBL Filtered
}

thresholds = np.linspace(0, 0.99, 300)

n_rows = len(datasets)   # 5
n_cols = len(profile_names)  # 3

# diagonal is only defined for first 3 rows
DIAGONAL = {(0,0), (1,1), (2,2)}

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 13),
                         sharex=True, sharey=True)

def get_rejected_grid_image(smiles_list, scores, profile_col_idx,
                             threshold=0.5, n=12, mols_per_row=4):
    """Draw a grid of molecules with score < threshold (rejected by own filter)."""
    rejected_idx = np.where(scores[:, profile_col_idx] < threshold)[0]
    # sample randomly from rejected
    rng = np.random.default_rng(42)
    pick = rng.choice(rejected_idx, size=min(n, len(rejected_idx)), replace=False)
    
    mols, legends = [], []
    for i in pick:
        smi = smiles_list[i]
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mols.append(mol)
            legends.append(f"{scores[i, profile_col_idx]:.2f}")
    
    if not mols:
        return None
    
    img = Draw.MolsToGridImage(
        mols, molsPerRow=mols_per_row,
        subImgSize=(200, 150),
        legends=legends,
        returnPNG=False   # returns PIL Image
    )
    return img


for row_i, (dataset_name, (score_matrix, _)) in enumerate(datasets.items()):
    for col_j, (profile_name, p_col) in enumerate(zip(profile_names, profile_col)):

        ax = axes[row_i, col_j]
        scores = score_matrix[:, p_col]
        scores = scores[np.isfinite(scores)]

        retention = np.array([(scores >= t).mean() for t in thresholds])
        y_intercept = (scores > 0).mean()

        is_diagonal = (row_i == col_j)
        # ChEMBL36 rows: no diagonal, but same color family as their counterpart
        is_validation = row_i >= 3

        color = COLORS[dataset_name]
        lw    = 2.2 if is_diagonal else 1.4
        alpha = 1.0 if is_diagonal else 0.7

        ax.plot(thresholds, retention, color=color, lw=lw, alpha=alpha)

        if is_diagonal:
            ax.fill_between(thresholds, retention, alpha=0.12, color=color)
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(1.8)
        elif is_validation:
            # dashed border to signal "test" rows
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(1.0)
                spine.set_linestyle("--")
        else:
            for spine in ax.spines.values():
                spine.set_edgecolor("#cccccc")
                spine.set_linewidth(0.8)

        # y-intercept line + label
        ax.axhline(y_intercept, color=color if (is_diagonal or is_validation) else "#aaaaaa",
                   lw=0.8, ls="--", alpha=0.6)
        ax.text(0.97, y_intercept + 0.03, f"{y_intercept:.2f}",
                transform=ax.transAxes, ha="right",
                fontsize=7.5, color=color if (is_diagonal or is_validation) else "#666666")

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.25))
        ax.xaxis.set_major_locator(mticker.MultipleLocator(0.25))
        ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.2, linewidth=0.5)

# --- Column headers ---
for col_j, name in enumerate(profile_names):
    axes[0, col_j].set_title(f"Profile: {name}", fontsize=10, fontweight="bold", pad=6)

# --- Row labels ---
for row_i, name in enumerate(datasets.keys()):
    suffix = "\n[val]" if row_i >= 3 else ""
    axes[row_i, 0].set_ylabel(f"{name}{suffix}\nFraction retained", fontsize=8.5)

# --- x-axis label ---
fig.text(0.5, -0.01, "LACAN score threshold", ha="center", fontsize=10)

plt.suptitle("LACAN Filter: Retention Curves across Profiles × Datasets",
             fontsize=12, y=1.01)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("lacan_retention_v2.pdf", bbox_inches="tight", dpi=150)
plt.show()


# ---------------------------------------------------------------
# SEPARATE FIGURE: rejected molecules on diagonal cells
# ---------------------------------------------------------------
diag_pairs = [
    ("ChEMBL Full",     chembl_full_scores,     chembl_full_smiles,     0),
    ("ChEMBL Filtered", chembl_filtered_scores, chembl_filtered_smiles, 1),
    ("COCONUT",         coconut_scores,          coconut_smiles,         2),
]

fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
fig2.suptitle("Molecules rejected by their own profile (score < 0.000000000001)", fontsize=12)

for ax, (name, score_matrix, smiles_list, p_col) in zip(axes2, diag_pairs):
    scores_arr = np.array(score_matrix)
    img = get_rejected_grid_image(smiles_list, scores_arr, p_col,
                                   threshold=0.000000000001, n=12, mols_per_row=4)
    if img is not None:
        ax.imshow(img)
    ax.set_title(f"{name}\nprofile rejects", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig("lacan_rejects.pdf", bbox_inches="tight", dpi=150)
plt.show()